# Singapore Jobs Analytics with DuckDB SQL

**Target users:** jobseekers and career switchers in Singapore.  
**Core question:** Which skills should I learn to improve employability?

This notebook uses a different approach from the pandas-first notebook:

1. Load the raw CSV into a local DuckDB database.
2. Perform cleaning, feature engineering, and analysis using SQL queries in Python.
3. Pull aggregated query results into Python only for matplotlib charts.

This is a good pattern for large CSV files because the full raw dataset does not need to be loaded directly into a pandas DataFrame.

## 1. Business Case

- **Scenario:** A career coach or career-switching platform wants to help jobseekers identify practical skills to learn.
- **Objective:** Compare skills by demand, salary, entry-level accessibility, and role pathway signals.
- **Target users:** jobseekers, career switchers, and career coaches.

The dashboard-style outputs answer:

1. Which skills show strong demand?
2. Which skills are associated with stronger salaries?
3. Which skills and role families are more entry-friendly?
4. What does progression from entry-level to senior-level look like?

## 2. Setup

DuckDB is the local analytical database. Pandas is used only to receive smaller aggregated SQL query outputs for display and plotting.

In [ ]:
from pathlib import Path
import warnings

import duckdb
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 80)
pd.set_option("display.max_rows", 80)
plt.style.use("seaborn-v0_8-whitegrid")

CSV_PATH = Path("SGJobData.csv") / "SGJobData.csv"
DB_PATH = Path("singapore_jobs.duckdb")

assert CSV_PATH.exists(), f"Could not find CSV at {CSV_PATH.resolve()}"
con = duckdb.connect(str(DB_PATH))
print(f"Connected to DuckDB database: {DB_PATH.resolve()}")

## 3. Load CSV into DuckDB

This creates a persistent table called `raw_jobs`. If it already exists, the notebook reuses it so the CSV does not need to be reloaded every time.

In [ ]:
raw_exists = con.execute("""
    SELECT COUNT(*)
    FROM information_schema.tables
    WHERE table_name = 'raw_jobs'
""").fetchone()[0]

if raw_exists == 0:
    con.execute(f"""
        CREATE TABLE raw_jobs AS
        SELECT
            categories,
            employmentTypes,
            metadata_jobPostId,
            metadata_newPostingDate,
            metadata_totalNumberJobApplication,
            metadata_totalNumberOfView,
            minimumYearsExperience,
            numberOfVacancies,
            positionLevels,
            postedCompany_name,
            salary_maximum,
            salary_minimum,
            salary_type,
            status_jobStatus,
            title,
            average_salary
        FROM read_csv_auto('{CSV_PATH.as_posix()}', header=true)
    """)
    print("Created raw_jobs from CSV")
else:
    print("raw_jobs already exists; using the existing DuckDB table")

row_count = con.execute("SELECT COUNT(*) FROM raw_jobs").fetchone()[0]
print(f"Rows in raw_jobs: {row_count:,}")

## 4. Raw Data Inspection

These checks document the table structure, salary type, and posting coverage before cleaning.

In [ ]:
display(con.execute("DESCRIBE raw_jobs").df())

display(con.execute("""
    SELECT salary_type, COUNT(*) AS postings
    FROM raw_jobs
    GROUP BY salary_type
    ORDER BY postings DESC
""").df())

display(con.execute("""
    SELECT
        MIN(TRY_CAST(metadata_newPostingDate AS DATE)) AS first_posting_date,
        MAX(TRY_CAST(metadata_newPostingDate AS DATE)) AS last_posting_date,
        COUNT(DISTINCT postedCompany_name) AS unique_companies,
        COUNT(DISTINCT metadata_jobPostId) AS unique_postings
    FROM raw_jobs
""").df())

## 5. SQL Cleaning and Feature Engineering

The cleaned table is created as `clean_jobs` inside DuckDB.

Feature engineering is done in SQL:

- posting date and posting month
- primary category from the first category object
- role family from title keywords
- seniority from title, position level, and minimum experience
- salary band
- competition per vacancy
- visible skill flag

**Limitation:** The CSV does not include full job descriptions, so skill analysis is based on visible title keywords.

In [ ]:
con.execute("DROP TABLE IF EXISTS clean_jobs")

con.execute(r'''
CREATE TABLE clean_jobs AS
WITH typed AS (
    SELECT
        categories,
        employmentTypes,
        metadata_jobPostId,
        TRY_CAST(metadata_newPostingDate AS DATE) AS posting_date,
        date_trunc('month', TRY_CAST(metadata_newPostingDate AS DATE)) AS posting_month,
        TRY_CAST(metadata_totalNumberJobApplication AS DOUBLE) AS applications,
        TRY_CAST(metadata_totalNumberOfView AS DOUBLE) AS views,
        TRY_CAST(minimumYearsExperience AS DOUBLE) AS minimum_years_experience,
        TRY_CAST(numberOfVacancies AS DOUBLE) AS number_of_vacancies,
        positionLevels,
        postedCompany_name,
        TRY_CAST(salary_maximum AS DOUBLE) AS salary_maximum,
        TRY_CAST(salary_minimum AS DOUBLE) AS salary_minimum,
        salary_type,
        status_jobStatus,
        title,
        TRY_CAST(average_salary AS DOUBLE) AS average_salary,
        lower(coalesce(title, '')) AS title_lc,
        lower(coalesce(positionLevels, '')) AS level_lc
    FROM raw_jobs
), engineered AS (
    SELECT
        *,
        COALESCE(json_extract_string(categories, '$[0].category'), 'Unknown') AS primary_category,
        CASE
            WHEN regexp_matches(title_lc, 'data analyst|data scientist|analytics|business intelligence|bi analyst|machine learning|ai engineer|data engineer') THEN 'Data & Analytics'
            WHEN regexp_matches(title_lc, 'software|developer|programmer|full stack|frontend|backend|java|\.net|mobile app|web developer') THEN 'Software Engineering'
            WHEN regexp_matches(title_lc, 'cyber|security analyst|soc analyst|information security|penetration|governance risk|grc') THEN 'Cybersecurity'
            WHEN regexp_matches(title_lc, 'cloud|devops|site reliability|sre|platform engineer|infrastructure engineer') THEN 'Cloud / DevOps'
            WHEN regexp_matches(title_lc, 'product manager|project manager|scrum master|programme manager|program manager|business project') THEN 'Product / Project'
            WHEN regexp_matches(title_lc, 'business analyst|strategy|consultant|transformation|process analyst') THEN 'Business / Strategy'
            WHEN regexp_matches(title_lc, '\bux\b|\bui\b|designer|figma|product design|graphic designer|visual designer') THEN 'UX / Design'
            WHEN regexp_matches(title_lc, 'digital marketing|seo|sem|performance marketing|social media|content marketer') THEN 'Digital Marketing'
            WHEN regexp_matches(title_lc, 'account|audit|finance|tax|treasury|payroll|bookkeeper') THEN 'Finance / Accounting'
            WHEN regexp_matches(title_lc, 'human resource|hr |recruit|talent acquisition|learning and development') THEN 'HR / Talent'
            WHEN regexp_matches(title_lc, 'sales|business development|customer service|account manager|relationship manager') THEN 'Sales / Customer'
            WHEN regexp_matches(title_lc, 'operations|admin|administrator|office manager|coordinator|executive assistant') THEN 'Operations / Admin'
            WHEN regexp_matches(title_lc, 'logistic|supply chain|warehouse|procurement|purchasing|planner') THEN 'Supply Chain / Logistics'
            WHEN regexp_matches(title_lc, 'engineer|technician|mechanical|electrical|civil|manufacturing|maintenance') THEN 'Engineering / Technical'
            WHEN regexp_matches(title_lc, 'nurse|clinic|healthcare|pharmac|laboratory|scientist|therapist') THEN 'Healthcare / Life Sciences'
            WHEN regexp_matches(title_lc, 'teacher|trainer|lecturer|educator|curriculum|instructor') THEN 'Education / Training'
            ELSE 'Other / General'
        END AS role_family,
        CASE
            WHEN regexp_matches(title_lc, 'intern|trainee|graduate|fresh|entry') OR level_lc LIKE '%fresh%' OR level_lc LIKE '%junior%' THEN 'Entry-level'
            WHEN minimum_years_experience <= 1 THEN 'Entry-level'
            WHEN regexp_matches(title_lc, 'senior|lead|principal|manager|head|director|architect') THEN 'Senior-level'
            WHEN level_lc LIKE '%senior%' OR level_lc LIKE '%manager%' OR level_lc LIKE '%management%' THEN 'Senior-level'
            WHEN minimum_years_experience >= 5 THEN 'Senior-level'
            ELSE 'Mid-level'
        END AS seniority,
        CASE
            WHEN average_salary < 3000 THEN '<$3k'
            WHEN average_salary < 5000 THEN '$3k-$5k'
            WHEN average_salary < 8000 THEN '$5k-$8k'
            WHEN average_salary < 12000 THEN '$8k-$12k'
            ELSE '$12k+'
        END AS salary_band,
        applications / NULLIF(number_of_vacancies, 0) AS competition_per_vacancy,
        CASE
            WHEN regexp_matches(title_lc, '\bpython\b|\bsql\b|database|power\s*bi|powerbi|tableau|analytics|machine learning|\bai\b|java|javascript|react|angular|vue|node|\.net|c#|cloud|aws|azure|gcp|devops|docker|kubernetes|cyber|\bsap\b|salesforce|project management|scrum|agile|accounting|audit|tax|seo|sem|figma|\bux\b|\bui\b') THEN 1
            ELSE 0
        END AS has_visible_skill
    FROM typed
)
SELECT *
FROM engineered
WHERE salary_type = 'Monthly'
  AND average_salary BETWEEN 500 AND 50000
  AND salary_minimum BETWEEN 0 AND 50000
  AND salary_maximum BETWEEN 0 AND 70000
  AND posting_date IS NOT NULL
''')

clean_count = con.execute("SELECT COUNT(*) FROM clean_jobs").fetchone()[0]
print(f"Rows in clean_jobs: {clean_count:,}")

## 6. Data Quality Snapshot

These SQL outputs support the assignment's data handling and process section.

In [ ]:
display(con.execute("""
    SELECT 'Rows loaded' AS metric, COUNT(*)::VARCHAR AS value FROM raw_jobs
    UNION ALL SELECT 'Rows after cleaning', COUNT(*)::VARCHAR FROM clean_jobs
    UNION ALL SELECT 'First posting date', MIN(posting_date)::VARCHAR FROM clean_jobs
    UNION ALL SELECT 'Last posting date', MAX(posting_date)::VARCHAR FROM clean_jobs
    UNION ALL SELECT 'Unique companies', COUNT(DISTINCT postedCompany_name)::VARCHAR FROM clean_jobs
    UNION ALL SELECT 'Postings with visible skill tag', SUM(has_visible_skill)::VARCHAR FROM clean_jobs
    UNION ALL SELECT 'Share with visible skill tag', ROUND(100.0 * AVG(has_visible_skill), 1)::VARCHAR || '%' FROM clean_jobs
""").df())

display(con.execute("""
    SELECT primary_category, COUNT(*) AS postings
    FROM clean_jobs
    GROUP BY primary_category
    ORDER BY postings DESC
    LIMIT 20
""").df())

## 7. Skill Mentions Table in SQL

The `skill_mentions` table contains one row per visible skill mention per job posting. This table is created entirely with SQL regular-expression rules.

In [ ]:
con.execute("DROP TABLE IF EXISTS skill_mentions")

skill_rules = [
    ("Python", r"\bpython\b"),
    ("SQL", r"\bsql\b|database|data warehouse"),
    ("Excel", r"\bexcel\b|spreadsheet"),
    ("Power BI", r"power\s*bi|powerbi"),
    ("Tableau", r"tableau"),
    ("Data Analytics", r"data analy|analytics|business intelligence|\bbi\b"),
    ("Machine Learning / AI", r"machine learning|\bai\b|artificial intelligence|deep learning|nlp"),
    ("Java", r"\bjava\b"),
    ("JavaScript", r"javascript|\bjs\b|react|angular|vue|node"),
    (".NET / C#", r"\.net|c#|c sharp|asp\.net"),
    ("Cloud", r"cloud|aws|azure|gcp|google cloud"),
    ("AWS", r"\baws\b|amazon web services"),
    ("Azure", r"azure"),
    ("DevOps", r"devops|ci/cd|cicd|site reliability|sre"),
    ("Docker / Kubernetes", r"docker|kubernetes|\bk8s\b|container"),
    ("Cybersecurity", r"cyber|information security|network security|penetration|soc analyst|grc"),
    ("SAP", r"\bsap\b"),
    ("Salesforce", r"salesforce"),
    ("Project Management", r"project management|project manager|pmp|programme manager|program manager"),
    ("Agile / Scrum", r"agile|scrum"),
    ("Accounting", r"accounting|audit|tax|bookkeeping|accounts payable|accounts receivable"),
    ("Digital Marketing", r"digital marketing|seo|sem|performance marketing|social media"),
    ("UI/UX / Figma", r"\bux\b|\bui\b|figma|product design"),
    ("Communication", r"communication|stakeholder|presentation"),
    ("Mandarin", r"mandarin|chinese speaking"),
]

selects = []
for skill, pattern in skill_rules:
    safe_skill = skill.replace("'", "''")
    safe_pattern = pattern.replace("'", "''")
    selects.append(f"""
        SELECT
            metadata_jobPostId, title, role_family, seniority, posting_month,
            average_salary, applications, views, number_of_vacancies,
            competition_per_vacancy, '{safe_skill}' AS skill
        FROM clean_jobs
        WHERE regexp_matches(title_lc, '{safe_pattern}')
    """)

con.execute("CREATE TABLE skill_mentions AS " + " UNION ALL ".join(selects))
print(f"Rows in skill_mentions: {con.execute('SELECT COUNT(*) FROM skill_mentions').fetchone()[0]:,}")

## 8. Market Overview Dashboard

These charts query aggregate results from DuckDB and plot them using matplotlib.

In [ ]:
role_counts = con.execute("""
    SELECT role_family, COUNT(*) AS postings
    FROM clean_jobs
    GROUP BY role_family
    ORDER BY postings DESC
    LIMIT 12
""").df().sort_values("postings")

seniority_counts = con.execute("""
    SELECT seniority, COUNT(*) AS postings
    FROM clean_jobs
    GROUP BY seniority
""").df()

salary_sample = con.execute("""
    SELECT average_salary
    FROM clean_jobs
    WHERE average_salary <= (SELECT quantile_cont(average_salary, 0.99) FROM clean_jobs)
""").df()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].barh(role_counts["role_family"], role_counts["postings"], color="#2a9d8f")
axes[0].set_title("Top role families by postings")
axes[0].set_xlabel("Postings")

order = ["Entry-level", "Mid-level", "Senior-level"]
seniority_counts["seniority"] = pd.Categorical(seniority_counts["seniority"], categories=order, ordered=True)
seniority_counts = seniority_counts.sort_values("seniority")
axes[1].bar(seniority_counts["seniority"].astype(str), seniority_counts["postings"], color=["#4c78a8", "#f58518", "#e45756"])
axes[1].set_title("Entry vs mid vs senior opportunities")
axes[1].tick_params(axis="x", rotation=25)

axes[2].hist(salary_sample["average_salary"], bins=60, color="#6c757d")
axes[2].set_title("Monthly salary distribution")
axes[2].set_xlabel("Average monthly salary")

plt.tight_layout()
plt.show()

## 9. Skill Summary and Employability Score

The score is intentionally simple and explainable:

- 50% demand: number of postings mentioning the skill
- 30% salary: median monthly salary
- 20% accessibility: entry-level share

In [ ]:
skill_summary = con.execute("""
WITH base AS (
    SELECT
        skill,
        COUNT(DISTINCT metadata_jobPostId) AS postings,
        median(average_salary) AS median_salary,
        avg(average_salary) AS average_salary,
        SUM(CASE WHEN seniority = 'Entry-level' THEN 1 ELSE 0 END) AS entry_postings,
        SUM(CASE WHEN seniority = 'Senior-level' THEN 1 ELSE 0 END) AS senior_postings,
        AVG(views) AS avg_views,
        AVG(applications) AS avg_applications,
        AVG(competition_per_vacancy) AS competition_per_vacancy
    FROM skill_mentions
    GROUP BY skill
), scored AS (
    SELECT
        *,
        entry_postings * 1.0 / NULLIF(postings, 0) AS entry_share,
        senior_postings * 1.0 / NULLIF(postings, 0) AS senior_share
    FROM base
), scaled AS (
    SELECT
        *,
        (postings - MIN(postings) OVER ()) * 1.0 / NULLIF(MAX(postings) OVER () - MIN(postings) OVER (), 0) AS postings_scaled,
        (median_salary - MIN(median_salary) OVER ()) * 1.0 / NULLIF(MAX(median_salary) OVER () - MIN(median_salary) OVER (), 0) AS median_salary_scaled,
        (entry_share - MIN(entry_share) OVER ()) * 1.0 / NULLIF(MAX(entry_share) OVER () - MIN(entry_share) OVER (), 0) AS entry_share_scaled
    FROM scored
)
SELECT
    skill,
    postings,
    ROUND(median_salary, 0) AS median_salary,
    ROUND(entry_share, 3) AS entry_share,
    ROUND(senior_share, 3) AS senior_share,
    ROUND(avg_views, 1) AS avg_views,
    ROUND(avg_applications, 1) AS avg_applications,
    ROUND(competition_per_vacancy, 1) AS competition_per_vacancy,
    ROUND(0.50 * COALESCE(postings_scaled, 0) + 0.30 * COALESCE(median_salary_scaled, 0) + 0.20 * COALESCE(entry_share_scaled, 0), 3) AS employability_score
FROM scaled
ORDER BY employability_score DESC
""").df()

skill_summary.head(20)

In [ ]:
top_demand = skill_summary.sort_values("postings", ascending=False).head(15).sort_values("postings")
top_salary = skill_summary[skill_summary["postings"] >= 50].sort_values("median_salary", ascending=False).head(15).sort_values("median_salary")

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
axes[0].barh(top_demand["skill"], top_demand["postings"], color="#2a9d8f")
axes[0].set_title("Most visible skill demand")
axes[0].set_xlabel("Postings mentioning skill")

axes[1].barh(top_salary["skill"], top_salary["median_salary"], color="#f58518")
axes[1].set_title("Highest median salary skills, min 50 postings")
axes[1].set_xlabel("Median monthly salary")

plt.tight_layout()
plt.show()

## 10. Salary Trends Over Time

Trend data is calculated in SQL and plotted in Python.

In [ ]:
top_skill_list = skill_summary.head(8)["skill"].tolist()
skill_filter = ", ".join(["'" + s.replace("'", "''") + "'" for s in top_skill_list])

monthly_skill_salary = con.execute(f"""
    SELECT
        posting_month,
        skill,
        COUNT(DISTINCT metadata_jobPostId) AS postings,
        median(average_salary) AS median_salary
    FROM skill_mentions
    WHERE skill IN ({skill_filter})
    GROUP BY posting_month, skill
    HAVING postings >= 5
    ORDER BY posting_month, skill
""").df()

fig, ax = plt.subplots(figsize=(14, 6))
for skill, group in monthly_skill_salary.groupby("skill"):
    ax.plot(group["posting_month"], group["median_salary"], marker="o", linewidth=2, label=skill)
ax.set_title("Monthly median salary trend for selected skills")
ax.set_xlabel("Posting month")
ax.set_ylabel("Median monthly salary")
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

## 11. Entry-Level Accessibility

This helps career switchers avoid choosing skills that are highly paid but mostly senior-heavy.

In [ ]:
entry_view = (
    skill_summary[skill_summary["postings"] >= 50]
    .sort_values("entry_share", ascending=False)
    .head(15)
    .sort_values("entry_share")
)

fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(entry_view["skill"], entry_view["entry_share"], color="#4c78a8")
ax.set_title("Skills with stronger entry-level accessibility")
ax.set_xlabel("Share of postings classified as entry-level")
ax.xaxis.set_major_formatter(lambda x, pos: f"{x:.0%}")
plt.tight_layout()
plt.show()

entry_view[["skill", "postings", "median_salary", "entry_share", "senior_share"]]

## 12. Role Pathways Using SQL

This section summarises entry, mid, and senior opportunities by role family.

In [ ]:
pathway_summary = con.execute("""
WITH skill_rank AS (
    SELECT
        c.role_family,
        c.seniority,
        s.skill,
        COUNT(*) AS mentions,
        ROW_NUMBER() OVER (PARTITION BY c.role_family, c.seniority ORDER BY COUNT(*) DESC, s.skill) AS skill_rank
    FROM clean_jobs c
    LEFT JOIN skill_mentions s USING (metadata_jobPostId)
    WHERE s.skill IS NOT NULL
    GROUP BY c.role_family, c.seniority, s.skill
), pathway AS (
    SELECT
        role_family,
        seniority,
        COUNT(DISTINCT metadata_jobPostId) AS postings,
        median(average_salary) AS median_salary
    FROM clean_jobs
    GROUP BY role_family, seniority
), top_skills AS (
    SELECT
        role_family,
        seniority,
        string_agg(skill, ', ' ORDER BY skill_rank) AS top_visible_skills
    FROM skill_rank
    WHERE skill_rank <= 5
    GROUP BY role_family, seniority
)
SELECT
    p.role_family,
    p.seniority,
    p.postings,
    ROUND(p.median_salary, 0) AS median_salary,
    COALESCE(t.top_visible_skills, 'No visible skill tags in title') AS top_visible_skills
FROM pathway p
LEFT JOIN top_skills t USING (role_family, seniority)
ORDER BY p.role_family, p.seniority
""").df()

pathway_summary.head(30)

In [ ]:
def show_role_pathway_sql(role_family="Data & Analytics"):
    subset = pathway_summary[pathway_summary["role_family"].eq(role_family)].copy()
    order = ["Entry-level", "Mid-level", "Senior-level"]
    subset["seniority"] = pd.Categorical(subset["seniority"], categories=order, ordered=True)
    subset = subset.sort_values("seniority")
    display(subset)

    if subset.empty:
        print("No pathway data for this role family.")
        return

    fig, ax1 = plt.subplots(figsize=(9, 5))
    ax1.bar(subset["seniority"].astype(str), subset["postings"], color="#2a9d8f", alpha=0.75)
    ax1.set_ylabel("Postings")
    ax1.set_title(f"Role pathway: {role_family}")
    ax2 = ax1.twinx()
    ax2.plot(subset["seniority"].astype(str), subset["median_salary"], color="#e45756", marker="o", linewidth=3)
    ax2.set_ylabel("Median monthly salary")
    plt.tight_layout()
    plt.show()

show_role_pathway_sql("Data & Analytics")

## 13. SQL-Powered Dashboard Function

This function keeps the dashboard logic database-first: filters are SQL filters, aggregation happens in DuckDB, and matplotlib handles the display.

In [ ]:
def sql_quote(value):
    return "'" + str(value).replace("'", "''") + "'"


def career_dashboard_sql(role_family="All", seniority="All", top_n=15):
    filters = ["1=1"]
    if role_family != "All":
        filters.append(f"role_family = {sql_quote(role_family)}")
    if seniority != "All":
        filters.append(f"seniority = {sql_quote(seniority)}")
    where_clause = " AND ".join(filters)
    c_where_clause = " AND ".join(["c." + f if f != "1=1" else f for f in filters])

    metrics = con.execute(f"""
        SELECT COUNT(*) AS postings, median(average_salary) AS median_salary, AVG(competition_per_vacancy) AS avg_applications_per_vacancy
        FROM clean_jobs
        WHERE {where_clause}
    """).df()

    if metrics.loc[0, "postings"] == 0:
        print("No postings match the selected filters.")
        return

    roles = con.execute(f"""
        SELECT role_family, COUNT(*) AS postings
        FROM clean_jobs
        WHERE {where_clause}
        GROUP BY role_family
        ORDER BY postings DESC
        LIMIT {int(top_n)}
    """).df().sort_values("postings")

    skills = con.execute(f"""
        SELECT s.skill, COUNT(DISTINCT s.metadata_jobPostId) AS postings
        FROM skill_mentions s
        JOIN clean_jobs c USING (metadata_jobPostId)
        WHERE {c_where_clause}
        GROUP BY s.skill
        ORDER BY postings DESC
        LIMIT {int(top_n)}
    """).df().sort_values("postings")

    salary_by_level = con.execute(f"""
        SELECT seniority, median(average_salary) AS median_salary
        FROM clean_jobs
        WHERE {where_clause}
        GROUP BY seniority
    """).df()

    print(f"Postings: {int(metrics.loc[0, 'postings']):,}")
    print(f"Median monthly salary: SGD {metrics.loc[0, 'median_salary']:,.0f}")
    print(f"Average applications per vacancy: {metrics.loc[0, 'avg_applications_per_vacancy']:.1f}")

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    axes[0].barh(roles["role_family"], roles["postings"], color="#2a9d8f")
    axes[0].set_title("Role family demand")
    axes[0].set_xlabel("Postings")

    if len(skills):
        axes[1].barh(skills["skill"], skills["postings"], color="#4c78a8")
        axes[1].set_title("Top visible skills")
        axes[1].set_xlabel("Postings")
    else:
        axes[1].text(0.5, 0.5, "No visible skill tags", ha="center", va="center")
        axes[1].set_axis_off()

    order = ["Entry-level", "Mid-level", "Senior-level"]
    salary_by_level["seniority"] = pd.Categorical(salary_by_level["seniority"], categories=order, ordered=True)
    salary_by_level = salary_by_level.sort_values("seniority")
    colors = ["#4c78a8", "#f58518", "#e45756"][:len(salary_by_level)]
    axes[2].bar(salary_by_level["seniority"].astype(str), salary_by_level["median_salary"], color=colors)
    axes[2].set_title("Median salary by seniority")
    axes[2].tick_params(axis="x", rotation=25)

    plt.tight_layout()
    plt.show()

career_dashboard_sql(role_family="All", seniority="All", top_n=15)

## 14. Recommendations for Jobseekers and Career Switchers

The final recommendation table balances demand, salary, and accessibility.

In [ ]:
recommendations = skill_summary[["skill", "postings", "median_salary", "entry_share", "senior_share", "employability_score"]].copy()
recommendations["entry_share"] = (recommendations["entry_share"] * 100).round(1)
recommendations["senior_share"] = (recommendations["senior_share"] * 100).round(1)
recommendations.head(15)

In [ ]:
def make_learning_message(row):
    if row["entry_share"] >= 30:
        access = "good entry-level accessibility"
    elif row["entry_share"] >= 15:
        access = "moderate entry-level accessibility"
    else:
        access = "mostly mid/senior positioning"
    return f"Learn {row['skill']}: {row['postings']:,} visible postings, median salary about SGD {row['median_salary']:,.0f}, {access}."

for _, row in recommendations.head(10).iterrows():
    print("- " + make_learning_message(row))

## 15. Written Report Draft

### Business case

- This solution supports jobseekers and career switchers deciding which skills to learn for better employability.
- It compares skills by demand, salary, entry-level accessibility, and role pathways.
- The intended user is a learner or career coach who needs evidence-backed guidance rather than generic course recommendations.

### Data handling and process

- Loaded the large CSV into a local DuckDB database instead of loading the full file directly into pandas.
- Created `raw_jobs`, `clean_jobs`, and `skill_mentions` tables.
- Cleaned salary data by keeping monthly salary postings with plausible salary ranges.
- Created SQL features for posting month, primary category, role family, seniority, salary band, and competition per vacancy.
- Extracted visible skill tags from job titles using SQL regular expressions.
- Queried aggregated outputs into Python only for plotting and presentation.

### Dashboard / app walkthrough

- Overview: top role families, seniority mix, and salary distribution.
- Skill demand: skills most visible in job titles.
- Salary: median salary by skill and monthly trend charts.
- Accessibility: entry-level share by skill.
- Pathways: role-family progression from entry to senior level.
- Filtered dashboard: SQL-powered view by role family and seniority.

### Challenges and learnings

- DuckDB is effective for large CSV analysis because SQL transformations can happen close to the data.
- The absence of full job descriptions limits skill extraction to visible title keywords.
- Salary fields need filtering before comparison.
- The most useful recommendation combines demand, pay, and entry-level access rather than optimising for only one metric.